# 02 -- Windows and export

Extract one sensor over a time window and export the record (acceleration), spectra and everything computed to a self-describing .h5.

In [ ]:
import sys
sys.path.insert(0, "../src")          # run from examples/

from asdea_sensors import SensorDataset
from asdea_sensors.config import settings

DATA = r"C:\Users\ppala\Desktop\02_31MAY2026"   # folder with the .h5 files

In [ ]:
ds = SensorDataset(path=DATA, date_source="filename", load_mode="auto", verbose=True)
ds.verbose  = True      # internal prints, ShakerMakerResults style
ds.n_jobs   = 4
ds.parallel = True
ds.devices

## Window by start + length

In [ ]:
w1 = ds.MOF00135.window(start="2026-05-31 15:00:00", length="60min")
w2 = ds.MOF00135.window(start="2026-05-31 18:30:00", length="300sec")
w3 = ds.MOF00135.window(start="2026-05-31 14:52:00", length="2hour")
w4 = ds.MOF00135.get_window(t0="2026-05-31 15:00", t1="2026-05-31 16:00")

## Process the window and compute several things

In [ ]:
win = ds.MOF00135.window("2026-05-31 15:00", "30min")
sig = win.signal().baseline().filter(0.1, 24.9).derive()
win.newmark(component="all")
win.arias(component="all")
win.fourier(component="x", smooth="konno")
win.psd(component="x")

## Export the window with everything (Provenance keeps it replicable)

In [ ]:
win.export_h5("MOF00135_15-1530.h5")    # acc record + spectra + all cached results + Provenance
# whole dataset (all sensors)
ds.export_h5("run_31MAY.h5")

## Read it back

In [ ]:
from asdea_sensors.io.results_file import ResultsFile
r = ResultsFile("MOF00135_15-1530.h5")
r.provenance              # input files, pipeline, params, config + hash
r.analyses("MOF00135")
data, params = r.get("MOF00135", "newmark_x")